In [ ]:
!pip install datasets
from IPython.display import clear_output

from datasets import load_dataset
from huggingface_hub import notebook_login

clear_output()

In [ ]:
notebook_login()

In [ ]:
!pip3 install openai
from openai import OpenAI
clear_output()

In [ ]:
ds = load_dataset('Z-Jafari/PersianQuAD')

In [ ]:
def make_prompt(question):
  return f"""In a Question-Answering system, is below question, a factoid question?
{question}
--Only 1 if yes
--Only 0 if no
  """

In [ ]:
prompts = []
for d in ds['train']:
  prompts.append(make_prompt(d['question']))


In [ ]:
prompts[100]

In [ ]:
len(prompts)

In [ ]:
import json

In [ ]:
api_key = = input("input API key: ")

In [ ]:
import asyncio
from openai import AsyncOpenAI

# Initialize Async Client
client = AsyncOpenAI(api_key=api_key, base_url="https://api.deepseek.com/v1")

async def classify_prompt(prompt):
    try:
        response = await client.chat.completions.create(
            model="deepseek-v4-flash",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=3,
            stream=False,
            # Optional: Disable thinking to get the answer faster in "Instant Mode"
            extra_body={"thinking": {"type": "disabled"}}
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

async def process_all_in_chunks(prompts, chunk_size=100):
    all_results = []

    # Slice the list: 0-100, 100-200, etc.
    for i in range(0, len(prompts), chunk_size):
        chunk = prompts[i : i + chunk_size]

        # Create tasks only for this specific chunk
        tasks = [classify_prompt(c) for c in chunk]

        print(f"Processing items {i} to {i + len(chunk)}...")
        chunk_results = await asyncio.gather(*tasks)

        all_results.extend(chunk_results)

    return all_results
# Example usage
# results = asyncio.run(process_all(your_list_of_20000_prompts))

In [ ]:
result = await process_all_in_chunks(prompts, 100)

In [ ]:
result

In [ ]:
len(result)

In [ ]:
ds['train'][0]['question']

In [ ]:
type(ds['train'])

In [ ]:
combined_zip = zip(ds['train'], result)

In [ ]:
# Assuming 'original_dicts' is your list of dictionaries
# and 'deepseek_results' is the list of 1/0 outputs.


# Create the new list of dictionaries
updated_list = []
for data_dict, result in combined_zip:
    # Create a copy so you don't modify the original list by accident
    new_dict = data_dict.copy()
    new_dict['is_factoid'] = result
    updated_list.append(new_dict)

# OR the "Pythonic" one-liner:
# updated_list = [{**d, 'is_factoid': r} for d, r in zip(original_dicts, deepseek_results)]

In [ ]:
len(updated_list)

In [ ]:
with open('train.json', 'w') as f:
      f.write(json.dumps(updated_list))

In [ ]:
from huggingface_hub import create_repo

In [ ]:
repoName = 'PersianQuAD_Factoid_Test'

repo_id = 'Z-Jafari/' + repoName

In [ ]:
create_repo(repo_id, repo_type="dataset")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj = 'train.json',
    path_in_repo='train.json',
    repo_id= repo_id,
    repo_type="dataset",
)

api.upload_file(
    path_or_fileobj = 'test.json',
    path_in_repo='test.json',
    repo_id= repo_id,
    repo_type="dataset",
)